# Creating the parquet dataset from SQLite tables

In [8]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

!pwd

%load_ext autoreload
%autoreload 2

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/data/3_build_fine_tuning_datasets/Study3_MultiMorbidity/stratified_dataset
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import torch
from hydra import compose, initialize
from omegaconf import OmegaConf
import logging
import time
import pickle 

from FastEHR.dataloader import FoundationalDataModule
from FastEHR.database.collector import SQLiteDataCollector
from SurvivEHR.examples.data.study_criteria import multimorbidity_inclusion_method

torch.manual_seed(1337)

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"    # if more informative debugging statements are needed
print(f"Using device: {device}.")


Using device: cuda.


In [10]:
# load the configuration file, override any settings 
with initialize(version_base=None, config_path="../../../../modelling/SurvivEHR/confs", job_name="dataset_creation_notebook"):
    cfg = compose(config_name="config_CompetingRisk11M", overrides=[])

# Create new dataset 
cfg.data.path_to_ds = None
print(OmegaConf.to_yaml(cfg))

is_decoder: true
data:
  batch_size: 64
  unk_freq_threshold: 0.0
  min_workers: 12
  global_diagnoses: false
  repeating_events: true
  path_to_db: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/cprd.db
  path_to_ds: null
  meta_information_path: /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle
  subsample_training: null
experiment:
  type: pre-train
  project_name: SurvivEHR
  run_id: ${head.SurvLayer}PreTrain_small_${experiment.seed}
  fine_tune_id: null
  notes: null
  tags: null
  train: true
  test: true
  verbose: true
  seed: 1337
  log: true
  log_dir: /rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/FoundationModelOutput/
  ckpt_dir: /rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/FoundationModelOutput/checkpoints/
fine_tuning:
  fine_tune_outcomes: null
  custom_outcome_method:
    _target_: null
  custom_stratification_method:
    _target_: null
  use_cal

### We have already specified training-test-validation splits for general practices in the UK

We must ensure we re-use the same splits to avoid data-leakage. This was done for the stratified pre-training dataset already. We can load those splits in here

### We now want to divide this existing training cohort into sub-populations

In [5]:
save_path = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/"

### Build the dataset for each of these collections of these general practices splits

In [7]:
# Build for each group 

for auth_group in [["London"], 
                   ["North East"]]:

    path_to_ds = save_path + f"MM_{'_'.join(auth_group)}/"
    path_to_split = save_path + f'practice_id_splits_{"_".join(auth_group)}.pickle'
    
    dm = FoundationalDataModule(path_to_db=cfg.data.path_to_db,
                                path_to_ds=path_to_ds,
                                load=True,
                                include_diagnoses=True,
                                include_measurements=True,
                                drop_missing_data=False,
                                drop_empty_dynamic=True,
                                tokenizer="tabular",
                                overwrite_practice_ids=path_to_split,
                                overwrite_meta_information=cfg.data.meta_information_path,
                                study_inclusion_method=multimorbidity_inclusion_method(),  # min_events=50
                                supervised=True,
                                num_threads=1
                               )

    vocab_size = dm.train_set.tokenizer.vocab_size
    
    print(f"{len(dm.train_set)} training patients")
    print(f"{len(dm.val_set)} validation patients")
    print(f"{len(dm.test_set)} test patients")
    print(f"{vocab_size} vocab elements")

INFO:root:Using meta information from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle


FileNotFoundError: [Errno 2] No such file or directory: '/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/MM_London/file_row_count_dict_train.pickle'

In [11]:
# dm.train_set.view_sample(1, max_dynamic_events=None, report_time=True)

In [10]:
for batch in dm.train_dataloader():
    break
print(batch.keys())

dict_keys(['static_covariates', 'tokens', 'ages', 'values', 'attention_mask', 'target_token', 'target_age_delta', 'target_value'])
